In [1]:
from openai import OpenAI

def generate_response(prompt: str, max_new_tokens: int = 128) -> str:
    client = OpenAI(
        base_url="https://openai.acemap.cn/v1",
        api_key="sk-BDfqjO9vKyWULLS89357F44d09Fb4b02B4BdFa9a79B183Ca"
    )
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_new_tokens,
    )
    return response.choices[0].message.content

In [10]:
task = "Extract marine lead data from scientific papers."

validation_prompt = """
Based on the expert's answer, extract structured knowledge related to the task.

Structured Knowledge Definition:
Background Knowledge: Key concepts, terminology, and context that form the foundation for understanding and completing the task.
Logical Constraints: Rules, conditions, and dependencies that must be followed.
Operational Guidelines: Clear steps, strategies, or heuristics for task execution and handling edge cases.
Validation Criteria: Measurable metrics or checks to ensure task quality and success.

Classify the knowledge into Background Knowledge, Logical Constraints, Operational Guidelines, or Validation Criteria if possible.

Then ask the expert for confirmation in the format:
Val:
[Structured knowledge here]. 
Is this correct?
"""

#### Collection Phase

In [ ]:
collection_prompt = """
You are an assistant helping to integrate data.
Data integration task: {task}
Follow this pipeline:
1. Generate a question to ask the expert, focusing on Background Knowledge, Logical Constraints, Operational Guidelines, or Validation Criteria, to obtain more information and complete the literature classification part of the data integration task.
2. Only output the question text.
"""

collection_questions = generate_response(collection_prompt.format(task=task), max_new_tokens=128)
print("Questions: ", collection_questions)

Questions:  What specific criteria or keywords should we use to classify a paper as relevant to marine lead data during the literature integration process?


In [ ]:
expert_answer = """
If we want to determine whether an article is related to extracting ocean lead data, there are several key points that need to be noted. Firstly, we focus on the element of lead itself, such as metallic lead or trace amounts of lead, rather than articles that primarily study the chemical synthesis of lead compounds or laboratory research. The background of the research is also important. We are interested in lead in the natural environment, such as its distribution, behavior in different systems, impact on ecosystems, or its cycling processes in land, ocean, atmosphere, and boundary systems.
Specifically, lead related papers can be classified into four categories: 'land', 'ocean', 'atmosphere', and 'boundary system'. The 'terrestrial' environment can include systems such as rivers, lakes, and soil, while the 'ocean' is not just the ocean, but also includes coastal areas, estuaries, sediments, and so on. The 'atmosphere' may involve the migration or deposition of lead in aerosols to the ocean or land. As for the 'boundary system', it refers to the place where two environments intersect, such as estuaries, wetlands, or the interface between the atmosphere and the ocean - where the migration and transformation of lead are also important. Among them, we want to focus on research papers on the 'ocean'.
On the other hand, we need to exclude some irrelevant articles. For example, articles that mainly study the chemical synthesis or industrial applications of lead do not need to be considered. Similarly, if an article only uses lead isotopes for dating, such as determining the age of rocks or archaeological artifacts, but does not address the role of lead in environmental processes, then it is not within our scope. The key is that the article should cover the role of lead in natural or environmental systems, especially in the context of earth science, marine science, or geology.
"""

validation_input = "User:\n" + collection_prompt + "\nAssistant:\n" + collection_questions + "\nExpert:\n" + expert_answer + "\nUser:\n" + validation_prompt
validation_response = generate_response(validation_input, max_new_tokens=512)
print(validation_response)

Val: 

Background Knowledge: 
- Focus on the element of lead, including metallic lead or trace amounts, in the natural environment.
- Lead-related papers are classified into four categories: 'land', 'ocean', 'atmosphere', and 'boundary system'.
- Relevant areas in the 'ocean' category include oceans, coastal areas, estuaries, and sediments.

Logical Constraints: 
- Exclude articles on chemical synthesis or industrial applications of lead.
- Exclude articles using lead isotopes solely for dating purposes without addressing environmental processes involving lead.

Operational Guidelines: 
- Focus on articles discussing the distribution, behavior, impact on ecosystems, or cycling processes of lead in natural environments, especially concerning earth science, marine science, or geology.

Validation Criteria: 
- Ensure articles cover the role of lead in natural or environmental systems, particularly in the context of marine science.

Is this correct?


#### Extraction Phase

In [ ]:
extraction_prompt = """
You are an assistant helping to integrate data.
Data integration task: {task}
Follow this pipeline:
1. Generate a question to ask the expert, focusing on Background Knowledge, Logical Constraints, Operational Guidelines, or Validation Criteria, to obtain more information and complete the data category classification part of the data integration task.
2. Only output the question text.
"""

extraction_questions = generate_response(extraction_prompt.format(task=task), max_new_tokens=128)
print("Questions: ", extraction_questions)

Questions:  What are the logical constraints or parameters that we should use to classify marine lead data effectively across different scientific papers?


In [ ]:
expert_answer = """
In our field, data integration is indeed a headache because the formats and terminologies vary so much across different papers. To ensure consistency and accuracy, we must first properly classify the data types. To put it simply, we need to establish a unified classification standard that a computer can understand.
I’ve given it some thought. For lead (Pb) data in the marine environment, there are three main categories that are most important. Everything else can, for now, be grouped into 'Other'.
The first category is the most fundamental: 'Lead Concentration,' which we can call Pb_CONC. This refers to the total amount of lead in a water body or sediment. But there's a key point here: lead exists in more than one form in the ocean. It can be 'Dissolved' or 'Particulate'. The dissolved form is what’s dissolved in the water, while the particulate form is adsorbed onto various suspended particles. These two are very different and have different scientific implications. So, when you're looking at a table, you must look for keywords like 'Dissolved' or 'Particulate', or similar descriptions such as 'D-Pb' or 'P-Pb'. Some older literature might not be as standardized, but they generally distinguish between these two forms. As long as a table contains this information and provides specific concentration values (with units like ng/L or pmol/kg), it falls into this 'Lead Concentration' category.
The second category is 'Lead-210 Isotope Concentration,' let's call it 210Pb_CONC. This is similar to the first category, but it focuses on a specific radioactive isotope of lead—Lead-210 (²¹⁰Pb). Lead-210 is especially important in oceanography; we often use it for sediment dating—essentially 'determining the age' of seafloor sediments—and for tracing the settling processes of particles. Therefore, it is also divided into 'Dissolved' and 'Particulate' phases. The criteria for identification are similar to the first category: look for explicit mentions of '²¹⁰Pb' or 'Lead-210' in the table, with data representing its activity or concentration (units might be dpm/100L, for example), and also pay attention to distinguishing between the 'Dissolved' and 'Particulate' forms.
The third category, which is also crucial, is the 'Stable Lead Isotope Ratios,' which can be called Pb_RATIO. This isn't about measuring concentration, but rather the ratio between different lead isotopes. For instance, the most common ratios we use are ²⁰⁶Pb/²⁰⁴Pb, ²⁰⁷Pb/²⁰⁴Pb, ²⁰⁸Pb/²⁰⁴Pb, as well as ²⁰⁶Pb/²⁰⁷Pb, ²⁰⁸Pb/²⁰⁶Pb, and ²⁰⁸Pb/²⁰⁷Pb. These ratios are like a 'fingerprint' for lead, helping us trace its source and determine whether it comes from natural sources (like rock weathering) or anthropogenic pollution (like gasoline or industrial emissions). When you're examining a data table, if you see column headers or data presented in the form of these ratios, you can be almost certain it belongs to this category. For isotope ratios, current research primarily focuses on the 'Dissolved' phase, as it better reflects the source characteristics of lead in the water column.
So, to summarize, you should first classify the data according to these three core categories. If the data in a table doesn't fit into lead concentration, lead-210 concentration, or those specific isotope ratios, then for now, just group it into the fourth category—'Other' (Other). 
"""

validation_input = "User:\n" + extraction_prompt + "\nAssistant:\n" + extraction_questions + "\nExpert:\n" + expert_answer + "\nUser:\n" + validation_prompt
validation_response = generate_response(validation_input, max_new_tokens=512)
print(validation_response)

Val:
Background Knowledge:
- Lead (Pb) exists as both 'Dissolved' and 'Particulate' forms in marine environments.
- Lead-210 (²¹⁰Pb) is a radioactive isotope used in sediment dating and particle tracing.
- Stable lead isotope ratios such as ²⁰⁶Pb/²⁰⁴Pb are used to trace the source of lead.

Logical Constraints:
- Lead concentrations must distinguish between 'Dissolved' and 'Particulate' forms and be measured in units like ng/L or pmol/kg.
- Lead-210 concentrations also need to differentiate between 'Dissolved' and 'Particulate' phases, often measured in units like dpm/100L.
- Lead isotope ratios primarily concern the 'Dissolved' phase and should match specific isotope ratio combinations.

Operational Guidelines:
- Classify data into three core categories: 'Lead Concentration' (Pb_CONC), 'Lead-210 Isotope Concentration' (210Pb_CONC), and 'Stable Lead Isotope Ratios' (Pb_RATIO).
- Use keywords like 'Dissolved,' 'Particulate,' '²¹⁰Pb,' 'Lead-210', and specific isotope ratios to classify d

#### Aggregation Phase

In [ ]:
# Using Pb concentrations as a demonstration
aggregation_prompt1 = """
You are an assistant helping to integrate data.
Data integration task: {task}
Follow this pipeline:
1. Generate a question to ask the expert, focusing on Background Knowledge, Logical Constraints, Operational Guidelines, or Validation Criteria, to discover what other pieces of information or related data fields must be linked with '{data_category}' data to make it complete and useful for the integration task.
2. Only output the question text.
"""

data_category = "Pb concentrations"
aggregation_questions = generate_response(aggregation_prompt1.format(task=task, data_category=data_category), max_new_tokens=128)
print("Questions: ", aggregation_questions)

Questions:  What related data fields or metadata (such as sample collection date, location coordinates, water depth, or sample type) should be linked with 'Pb concentrations' to ensure that the integrated data is comprehensive and contextually informative for marine lead studies?


In [ ]:
expert_answer = """
The data sources are incredibly diverse—some come from old papers published decades ago, others from the latest online databases, and their formats are all different. If you want to integrate this data, especially the core metric of 'Pb concentrations,' for systematic analysis, you absolutely must link it with several key pieces of metadata. Otherwise, the data is just a pile of meaningless numbers.
First and foremost, the most fundamental piece of information is the geographic location. Without location information, we don't know where the lead concentration was measured. This involves two dimensions: Longitude and Latitude.
When you organize the data, you must create two standard columns. For one, I'd suggest calling it 'Longitude' with the unit 'degrees_east'. You have to be particularly careful here because, in data processing, inconsistent formats are our biggest fear. In papers, you'll see notations like '30°E' or '110°W', which computers don't recognize. So, we need to establish a rule: all East longitudes (E) are represented by positive values, so 30°E becomes 30; all West longitudes (W) are negative, so 110°W becomes -110. This way, all longitude data becomes standardized.
The same principle applies to latitude. The other column should be named 'Latitude', with the unit 'degrees_north'. The rule is: North latitudes (N) are positive, and South latitudes (S) are negative. For example, 30°N is 30, and 45°S is -45. After this processing, you can directly plot the data in any mapping software or analysis tool, which is extremely convenient.
However, you'll often run into a frustrating problem: the table itself only contains concentration data, with no longitude or latitude. What do you do then? You have to act like a detective. This information is often hidden elsewhere, such as in the table's title, the footnotes below it, or in the paragraph in the main text describing the sampling procedure. It might even be in another figure or table within the same paper. You must dig out this information and match it up. If you really can't find it, then that data point, unfortunately, has to be discarded. For spatial distribution studies, data without location information is of little value.
Next, the third key dimension is DEPTH. The ocean is three-dimensional. Water bodies at different depths have completely different chemical environments and physical properties, and the sources and forms of lead can also vary. Therefore, there must be a 'DEPTH' column, with the unit standardized to 'm' (meters). This allows us to analyze the distribution patterns of lead on a vertical profile. Similarly, if the data is given as a range, like '10-20 meters', for the sake of consistency, we should just use the starting value, recording '10'. The priority is to make the data usable.
Finally, we get to our main subject—the Pb-related data. This is also complex because the 'lead' measured in different studies may not be the same thing. Some measure total dissolved lead, others particulate lead, or specific isotopes. Therefore, when you create your table, do not try to standardize them. Instead, preserve the original column names and their units. For example, if a paper has two columns, 'Dissolved Pb (μg/L)' and 'Particulate Pb (ng/g)', you should transfer both columns over exactly as they are. This way, during subsequent analysis, we can selectively use these different forms of lead data based on our research objectives.
"""

validation_input = "User:\n" + aggregation_prompt1 + "\nAssistant:\n" + aggregation_questions + "\nExpert:\n" + expert_answer + "\nUser:\n" + validation_prompt
validation_response = generate_response(validation_input, max_new_tokens=512)
print(validation_response)

Val:

Background Knowledge:
- Key metadata for 'Pb concentrations' includes Geographic Location (Longitude and Latitude), and Depth, which are critical for contextual understanding.
- Different studies may measure different forms of lead, such as total dissolved lead or particulate lead.

Logical Constraints:
- Longitude values must be standardized as positive for East (E) and negative for West (W).
- Latitude values must be standardized as positive for North (N) and negative for South (S).
- Depth should be represented in meters and should take the starting value if given as a range.

Operational Guidelines:
- Extract geographic location, depth, and lead form/units information meticulously from tables, figures, footnotes, or related text in original sources.
- Preserve original lead data column names and units during data integration.
- Discard data points if geographic location information cannot be found.

Validation Criteria:
- Ensure data with standardized longitude, latitude, and

In [19]:
# Standardize the units of Pb concentrations as a demonstration
aggregation_prompt2 = """
You are an assistant helping to integrate data.
Data integration task: {task}
Follow this pipeline:
1. Generate a question to ask the expert, focusing on Background Knowledge, Logical Constraints, Operational Guidelines, or Validation Criteria, to obtain more information about target standards for normalizing the units of '{data_category}' data. This includes understanding all potential source units and the required conversion procedures.
2. Only output the question text.
"""

data_category = "Pb concentrations"
aggregation_questions = generate_response(aggregation_prompt2.format(task=task, data_category=data_category), max_new_tokens=128)
print("Questions: ", aggregation_questions)

Questions:  What are the standard units for 'Pb concentrations' in marine lead data, and what conversion procedures should be applied to normalize data from common source units like µg/L, mg/L, or ppb?


In [ ]:
expert_answer = """
Regarding the unit standardization for 'lead (Pb) concentrations' in marine data. This is truly a critical issue and often a headache for everyone. When we're processing data from different cruises, different countries, or even different labs, the very first step is to unify the units. Otherwise, you can't compare them at all.
In our field, especially for those of us in marine geochemistry and environmental science, there's a common international practice now, which is to convert all lead concentrations ultimately to pmol/kg, which stands for 'picomoles per kilogram.'
You might ask, why this specific unit? Why not the more familiar µg/L (micrograms per liter) or ppb (parts per billion)? There are a few considerations behind this.
First, using 'moles' (mol) is a chemist's habit. It focuses on the amount of substance, the number of particles, rather than mass. This is more meaningful when studying the biogeochemical cycling of elements because it more directly reflects the stoichiometric relationships in chemical reactions. Lead is Pb, and its molar mass is about 207.2 g/mol, which is the basis for conversion.
Second, the denominator is 'kilogram' (kg) instead of 'liter' (L), meaning it's a mass-based concentration, not a volume-based one. This is a core practice in oceanography. Why? Because the density of seawater is not constant! It varies with temperature and salinity. If you measure a concentration in a bottle of 25°C surface seawater and another in 4°C deep-sea water, and you use 'per liter' for both, the 'substance' of that 'liter' itself is different. So, we standardize by using 'per kilogram of seawater' as the baseline, which eliminates errors caused by density variations. We typically use a standard average seawater density for conversion, like 1.025 kg/L, which is a very common approximation.
So, our goal is, no matter what varied units you get in the raw data—like nmol/kg, µg/L, mg/L, or even something like Bq/kg for radioactivity—we have to find a way to convert it to pmol/kg.
How do we do the conversion, specifically? It's basically high school chemistry and physics calculations, but you have to be careful.
The simplest case: If you're converting from nmol/kg (nanomoles per kilogram) to pmol/kg, you just multiply by 1000, because 1 nmol equals 1000 pmol. This is just a prefix conversion.
A bit more complex: For example, if you have µg/L (micrograms per liter). Step one, you need to convert the volume unit 'liter' to a mass unit 'kilogram'. You use the density we just talked about, multiplying by 1/1.025. The unit then becomes µg/kg. Step two, you need to convert the mass 'microgram' to the amount of substance 'mole'. You do this by dividing the value by lead's molar mass, 207.2 g/mol. But be careful, one is micrograms (µg) and the other is grams (g), there's a factor of 10^6 difference. Finally, you convert mol to pmol, which means multiplying by 10^12. After this whole process, you'll get pmol/kg.
Of course, when we write code or process data, we automate this process. You'd write a function that takes a value and its original unit as input and outputs the converted value.
But there's a very important prerequisite here: the 'dimensions' of the units must be compatible. For instance, Bq/kg is a unit of radioactive activity, describing the rate of nuclear decay, whereas pmol/kg is a concentration of the amount of substance. These two describe completely different physical meanings. There is no universal conversion factor between them, unless you have detailed information about the lead isotopes in that specific sample. So, in such cases, our function should simply return a 'null' value (None) to tell us 'this path is blocked.'
We also encounter some strange unit notations, like m³ (cubic meters) or 100L-1 (per 100 liters). When writing the function, these need to be considered as well, to correctly parse exponential or inverse forms.
To sum it all up, the core idea is: standardize to pmol/kg, perform conversions based on seawater density (1.025 kg/L) and lead's molar mass (207.2 g/mol), and be able to identify cases where units are dimensionally incompatible and cannot be converted. Only by doing this can we integrate global data to see a reliable and consistent big picture of lead distribution in the oceans.
"""

validation_input = "User:\n" + aggregation_prompt2 + "\nAssistant:\n" + aggregation_questions + "\nExpert:\n" + expert_answer + "\nUser:\n" + validation_prompt
validation_response = generate_response(validation_input, max_new_tokens=512)
print(validation_response)

Val:
- **Background Knowledge**:
  - The standard unit for lead (Pb) concentrations in marine data is picomoles per kilogram (pmol/kg).
  - The molar mass of lead is approximately 207.2 g/mol.
  - The density of seawater affects concentration measurements and generally approximates 1.025 kg/L.

- **Logical Constraints**:
  - Volume-based units (e.g., µg/L) should be first converted to mass-based units using the standard seawater density.
  - The dimensions of the original units must be compatible for conversion (e.g., concentration vs. radioactive activity).

- **Operational Guidelines**:
  - Convert volume-based units to mass-based units using a density of 1.025 kg/L.
  - Transform mass units into moles using the molar mass of lead, then adjust for the appropriate scale (e.g., µg to g and mol to pmol).
  - Use a function to automate the conversion process, handling different initial units and edge cases.

- **Validation Criteria**:
  - Ensure that all data is normalized to pmol/kg and